In [13]:
import os
import pandas as pd

In [14]:
df = pd.read_csv("/Users/hinahaq/Downloads/data_file_project5/final_test_data.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (317235, 7)


,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


## KPI 1: Completion Rate 

**Definition:** % of unique visits reaching `confirm` from `start`  
**Success criteria:** Test > Control by 5%  
**Why:** More completions = more revenue 

In [15]:
completion_rate = (
    df[df['process_step']=='confirm']['visit_id'].nunique() / 
    df[df['process_step']=='start']['visit_id'].nunique() * 100
)
print(f"Completion Rate: {completion_rate:.2f}%")

Completion Rate: 58.92%


In [16]:
completion_by_group = (
    df[df['process_step'] == 'confirm']
    .groupby('group_test')['visit_id'].nunique()
    / df[df['process_step'] == 'start'].groupby('group_test')['visit_id'].nunique()
    * 100
)

print(completion_by_group)

group_test
control    51.912003
test       65.539705
Name: visit_id, dtype: float64


## Task 1 Results 

**Test:** 65.54% completion  
**Control:** 51.91% completion  
**Improvement:** +13.63% (beats 5% threshold!)  

**Learned:** Test group design significantly increases conversions.

## KPI 4: Funnel Drop-off Rate

**Definition:** % users lost between consecutive steps  
**Success:** Test has flatter drop-off curve  
**Why:** Pinpoints problem steps

In [19]:
print("All process_step values:")
print(df['process_step'].value_counts())

All process_step values:
process_step
start      101153
step_1      68210
step_2      56672
step_3      48264
confirm     42936
Name: count, dtype: int64


In [22]:
print("Unique group_test values:")
print(df['group_test'].unique())
print("\nSample group_test:")
print(df['group_test'].value_counts())

Unique group_test values:
['test' 'control']

Sample group_test:
group_test
test       176699
control    140536
Name: count, dtype: int64


In [23]:
# Raw counts table
pivot = df[df['process_step'].isin(['start','step_1','step_2','step_3','confirm'])].groupby(['process_step','group_test'])['visit_id'].nunique().unstack(fill_value=0)

print("Unique visits per step:")
print(pivot)

print("\nDrop-off analysis:")
step_order = ['start','step_1','step_2','step_3','confirm']
for i in range(1, len(step_order)):
    prev = step_order[i-1]
    curr = step_order[i]
    test_drop = ((pivot.loc[prev, 'test'] - pivot.loc[curr, 'test']) / pivot.loc[prev, 'test'] * 100).round(1)
    control_drop = ((pivot.loc[prev, 'control'] - pivot.loc[curr, 'control']) / pivot.loc[prev, 'control'] * 100).round(1)
    print(f"{prev}→{curr}: Test {test_drop}% vs Control {control_drop}% (diff: {test_drop-control_drop:+.1f}%)")

Unique visits per step:
group_test    control   test
process_step                
confirm         16046  21731
start           30910  33157
step_1          23548  28285
step_2          20138  24503
step_3          18300  22186

Drop-off analysis:
start→step_1: Test 14.7% vs Control 23.8% (diff: -9.1%)
step_1→step_2: Test 13.4% vs Control 14.5% (diff: -1.1%)
step_2→step_3: Test 9.5% vs Control 9.1% (diff: +0.4%)
step_3→confirm: Test 2.1% vs Control 12.3% (diff: -10.2%)


## Task 4 Results
## Task 4: Funnel Drop-off Analysis

### Raw Unique Visits Per Step
| Step    | Control | Test  |
|---------|---------|-------|
| start   | 30,910  | 33,157|
| step_1  | 23,548  | 28,285|
| step_2  | 20,138  | 24,503|
| step_3  | 18,300  | 22,186|
| confirm | 16,046  | 21,731|

### Drop-off Between Steps
| Transition     | Control | Test  | Test Advantage |
|----------------|---------|-------|----------------|
| start→step_1   | 23.8%   | 14.7% | +9.1%         |
| step_1→step_2  | 14.5%   | 13.4% | +1.1%         |
| step_2→step_3  | 9.1%    | 9.5%  | -0.4%         |
| step_3→confirm | 12.3%   | 2.1%  | +10.2%        |

**Most Problematic:** start→step_1 (both groups)  
**Test Wins Biggest:** Early funnel + final conversion step

## KPI 5: Average Steps Reached

**Definition:** Mean furthest step per visit  
**Success:** Test > Control  
**Why:** Proxy for engagement

In [ ]:
step_map = {'start':1, 'step_1':2, 'step_2':3, 'step_3':4, 'confirm':5}
df['step_num'] = df['process_step'].map(step_map)

max_steps = df.groupby(['visit_id', 'group_test'])['step_num'].max()
avg_steps = max_steps.groupby('group_test').mean()

print("Avg furthest step reached:")
print(avg_steps.round(2))

Avg furthest step reached:
group_test
control    3.51
test       3.90
Name: step_num, dtype: float64


## Task 5 Results

**Test:** 3.90 steps (reaches step_3)  
**Control:** 3.51 steps (barely reaches step_3)  
**Improvement:** +0.39 steps  

**Learned:** Test users progress 11% further through funnel.